# **Segmentasi Profil Nasabah Perbankan Berdasarkan Pola Transaksi dan Karakteristik Finansial Menggunakan Metode K-Means dan Hierarchical Clustering**
---
## Kelompok Kimpul - 57
- Sukmawati Kharisma Gati - 123240137
- Aulya Revalina - 123240141
- Azzah Fauziya Kamila - 123240168
- Anindya Bintarti - 123240197


---
# 1. Import Library

Pada tahap ini dilakukan pemanggilan beberapa library Python
yang digunakan untuk membantu proses analisis dan pengolahan data.

Library yang digunakan meliputi:
- Pandas: mengolah dan memanipulasi data dalam bentuk DataFrame.
- NumPy: melakukan operasi numerik dan perhitungan matematis.
- Matplotlib: membuat visualisasi data seperti histogram dan grafik.
- Seaborn: membuat visualisasi statistik seperti boxplot dan heatmap.
- Scikit-learn: membantu proses preprocessing, khususnya feature scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

---
# 2. Memuat Dataset

Tahap ini dilakukan untuk membaca dataset yang akan digunakan
dalam penelitian.

Dataset yang digunakan adalah Bank Transaction Dataset for
Fraud Detection yang diperoleh dari Kaggle.

In [ ]:
df = pd.read_csv("bank_transactions_data_2.csv")

df.head()

---
# 3. Data Understanding

## 3.1 Memahami Struktur Dataset

Tahap ini bertujuan untuk mengetahui gambaran umum dataset,
meliputi jumlah baris, jumlah kolom, nama atribut, tipe data,
serta informasi jumlah data yang tersedia pada setiap kolom.

Pemahaman terhadap struktur dataset diperlukan untuk mengetahui
karakteristik data dan menentukan proses pengolahan yang sesuai.

In [ ]:
print("Ukuran dataset : ", df.shape)

df.info()

print(df.columns.tolist())

## 3.2 Statistik Deskriptif

Statistik deskriptif dilakukan untuk memperoleh gambaran
karakteristik data berdasarkan nilai statistiknya.

Analisis mencakup nilai rata-rata (mean), standar deviasi,
nilai minimum, kuartil, median, dan maksimum.

Hasil analisis digunakan untuk memahami rentang nilai,
persebaran data, serta perbedaan karakteristik antaratribut.

In [ ]:
df.describe()

In [ ]:
df.describe(include=['O'])

## 3.3 Pemeriksaan Missing Values

Missing values merupakan kondisi ketika suatu atribut
memiliki nilai yang kosong atau tidak tersedia.

Pemeriksaan dilakukan untuk mengetahui jumlah dan persentase
nilai kosong pada setiap atribut.

Hasil pemeriksaan menjadi dasar untuk menentukan apakah
diperlukan penanganan missing values, seperti penghapusan
data atau imputasi menggunakan nilai statistik tertentu.

In [ ]:
print("jumlah missing value perkolom : ")
print(df.isnull().sum())

## 3.4 Pemeriksaan Data Duplikat

Data duplikat merupakan data yang memiliki kesamaan pada
seluruh atribut atau baris tertentu.

Pemeriksaan dilakukan untuk mengetahui apakah terdapat
pencatatan data yang berulang.

Data duplikat perlu diperhatikan karena dapat menyebabkan
perhitungan statistik menjadi tidak akurat dan memengaruhi
hasil analisis clustering.

In [ ]:
print("Jumlah data duplikat :", df.duplicated().sum())

## 3.5 Analisis Distribusi Data

Analisis distribusi dilakukan untuk memahami pola persebaran
nilai pada setiap atribut.

Pada atribut numerik, distribusi diamati menggunakan histogram.
Sementara itu, atribut kategorikal dianalisis menggunakan
diagram batang (bar chart).

Visualisasi ini membantu mengidentifikasi pola distribusi,
ketidakseimbangan kategori, serta kemungkinan kemencengan
distribusi data (skewness).

In [ ]:
numerical_columns = [
    "JumlahNominalTransaksi",
    "UsiaNasabah",
    "DurasiTransaksi",
    "JumlahPercobaanLogin",
    "SaldoRekening"
]

df[numerical_columns].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

## 3.5 Identifikasi Outlier

Outlier merupakan nilai yang memiliki karakteristik berbeda
secara signifikan dibandingkan sebagian besar data lainnya.

Identifikasi outlier dilakukan menggunakan metode
Interquartile Range (IQR), yaitu metode yang menggunakan
jarak antara kuartil pertama (Q1) dan kuartil ketiga (Q3).

Batas bawah dan batas atas ditentukan menggunakan rumus:

IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 ├ù IQR

Upper Bound = Q3 + 1.5 ├ù IQR

Nilai yang berada di luar batas tersebut ditandai sebagai
kandidat outlier. Hasil identifikasi digunakan untuk
mempertimbangkan perlakuan terhadap outlier pada tahap
Data Preparation.

In [ ]:
for col in numerical_columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot {col}")
    plt.show()

## 3.6 Analisis Korelasi

Analisis korelasi dilakukan untuk mengetahui hubungan
antaratribut numerik.

Korelasi menunjukkan arah dan kekuatan hubungan linear
antara dua variabel.

Hasil analisis divisualisasikan menggunakan correlation
heatmap untuk membantu mengidentifikasi atribut yang
memiliki hubungan kuat.

Informasi ini dapat menjadi pertimbangan dalam proses
pemilihan fitur (feature selection).

In [ ]:
correlation = df[numerical_columns].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()

---
# 4. Data Prepatarion

## 4.1 Data Cleaning

Data Cleaning merupakan proses membersihkan dataset dari
permasalahan kualitas data yang ditemukan pada tahap
Data Understanding.

Proses ini dilakukan agar data yang digunakan memiliki
kualitas yang lebih baik sebelum memasuki tahap preprocessing.

Beberapa proses yang diperhatikan meliputi:

1. Menghapus data duplikat apabila ditemukan.
2. Menangani missing values apabila terdapat nilai kosong.
3. Memeriksa konsistensi data, termasuk format tanggal dan
   nilai atribut.
4. Mempertimbangkan penanganan outlier berdasarkan
   karakteristik dan kebutuhan penelitian.

Tidak semua data yang teridentifikasi sebagai outlier
harus dihapus. Penanganannya perlu mempertimbangkan
relevansi nilai tersebut terhadap tujuan clustering.

### 1. Penanganan data duplikat

_*Berdasarkan hasil data understanding, tidak ada data duplikat yg ditemukan._

### 2. Missing value 

_*Tidak ada missing value yang ditemukan._

### 3. Memeriksa konsistensi data, termasuk format tanggal dan nilai atribut.

In [ ]:
# Daftar kolom kategorikal utama, ini fitur2 yg kita pakai
cat_cols = ['JenisTransaksi', 'SaluranTransaksi', 'Lokasi', 'PekerjaanNasabah']

print("Pengecekan Nilai Unik pada Atribut Kategorikal")
for col in cat_cols:
    print(f"\nKolom: {col}")

    print(df[col].unique())

## 4.2 Data Preprocessing

Data Preprocessing merupakan proses mengubah data yang
telah dibersihkan agar dapat digunakan oleh algoritma
machine learning.

Pada penelitian ini, preprocessing disesuaikan dengan
karakteristik algoritma K-Means dan Hierarchical Clustering
yang menggunakan perhitungan jarak antarobservasi.

Tahapan preprocessing yang dipertimbangkan meliputi:

1. Penentuan unit analisis.
2. Pemilihan dan penyesuaian tipe data.
3. Encoding atribut kategorikal apabila digunakan.
4. Feature scaling untuk menyamakan skala antaratribut.

Proses preprocessing bertujuan untuk memastikan bahwa
setiap fitur dapat digunakan secara tepat dalam perhitungan
jarak pada proses clustering.

### 4.2.1 Penentuan Unit Analisis

Unit analisis merupakan objek yang akan dikelompokkan
oleh algoritma clustering.

Pada penelitian ini, unit analisis perlu ditentukan
sebagai transaksi atau akun nasabah.

Penentuan ini penting karena satu AccountID dapat memiliki
lebih dari satu catatan transaksi dan ditemukan
ketidakkonsistenan nilai CustomerAge pada beberapa akun.

Oleh karena itu, pembentukan unit analisis harus
mempertimbangkan konsistensi identitas akun agar hasil
segmentasi tidak menghasilkan profil yang keliru.

In [ ]:
# 1. Melihat total keseluruhan baris dan total unik akun nasabah (AccountID)
total_transaksi = len(df)
total_akun = df['IDRekening'].nunique() # Sesuaikan 'IDRekening' dengan nama kolom ID akun kamu

print(f"Total Baris (Catatan Transaksi): {total_transaksi}")
print(f"Total Akun Unik (AccountID): {total_akun}")
print(f"Rata-rata transaksi per akun: {total_transaksi / total_akun:.2f}")

# 2. Mengecek apakah ada AccountID yang memiliki inkonsistensi 'UsiaNasabah' (CustomerAge)
# Caranya: menghitung jumlah nilai unik usia untuk setiap IDRekening
age_check = df.groupby('IDRekening')['UsiaNasabah'].nunique()
inconsistent_accounts = age_check[age_check > 1]

print(f"\nJumlah akun dengan inkonsistensi usia nasabah: {len(inconsistent_accounts)}")

if len(inconsistent_accounts) > 0:
    print("Contoh IDRekening yang memiliki variasi usia berbeda:")
    print(inconsistent_accounts.head())
else:
    print("Aman! Semua akun memiliki catatan usia nasabah yang konsisten.")

### 4.2.2 Encoding Data Kategorikal

Encoding merupakan proses mengubah atribut kategorikal
menjadi representasi numerik agar dapat diproses oleh
algoritma machine learning.

Apabila atribut seperti TransactionType, Channel, dan
CustomerOccupation digunakan sebagai fitur, encoding
dapat dilakukan menggunakan metode One-Hot Encoding.

Metode ini merepresentasikan setiap kategori menjadi
kolom indikator bernilai 0 atau 1.

Encoding hanya dilakukan terhadap atribut kategorikal
yang dipilih untuk digunakan dalam clustering.

### 1. One-Hot Enoding untuk JenisTransaksi

In [ ]:
df_encoded_jenis = pd.get_dummies(df['JenisTransaksi'], prefix='Tipe')

print("Hasil Encoding - JenisTransaksi:")
display(df_encoded_jenis.head())

### 2. One-Hot Encoding Saluran Transaksi

In [ ]:
df_encoded_saluran = pd.get_dummies(df['SaluranTransaksi'], prefix='Kanal')

print("Hasil Encoding - SaluranTransaksi:")
display(df_encoded_saluran.head())

### 4.2.3 Agregasi 

In [ ]:
df_lengkap = pd.concat([df, df_encoded_jenis, df_encoded_saluran], axis=1)

print("Total baris transaksi awal:", len(df_lengkap))
print("Jumlah akun unik (IDRekening):", df_lengkap['IDRekening'].nunique())

# 2. Lakukan Agregasi menggunakan df_lengkap
df_agregasi = df_lengkap.groupby('IDRekening').agg({
    # Demografi: Mengatasi inkonsistensi usia dengan modus
    'UsiaNasabah': lambda x: x.mode()[0],
    
    # Profiling: Disimpan untuk analisis segmen
    'PekerjaanNasabah': 'first',
    
    # Fitur Numerik & Agregasinya
    'JumlahNominalTransaksi': ['sum', 'mean'], 
    'IDTransaksi': 'count',                        # Menghitung frekuensi transaksi per akun
    'DurasiTransaksi': 'mean',                      
    'JumlahPercobaanLogin': 'mean',             
    'SaldoRekening': 'mean',
    
    # Fitur Hasil Encoding (Dijumlahkan per akun)
    'Tipe_Debit': 'sum',
    'Tipe_Credit': 'sum',
    'Kanal_ATM': 'sum',
    'Kanal_Branch': 'sum',
    'Kanal_Online': 'sum'
}).reset_index()

# 3. Merapikan nama kolom hasil multi-index
df_agregasi.columns = [
    'ID_Rekening', 'Usia_Nasabah', 'Pekerjaan_Nasabah', 
    'Total_Nominal_Transaksi', 'Rata_Rata_Nominal', 
    'Frekuensi_Transaksi', 'Rata_Rata_Durasi', 
    'Rata_Rata_Login', 'Rata_Rata_Saldo',
    'Frekuensi_Debit', 'Frekuensi_Kredit',
    'Kanal_ATM', 'Kanal_Branch', 'Kanal_Online'
]

# 4. Validasi hasil akhir (harus 495 baris)
print("Total baris setelah agregasi:", len(df_agregasi))
display(df_agregasi.head())

## 4.3 Feature Selection

Feature Selection merupakan proses memilih atribut
yang paling relevan dengan tujuan penelitian.

Tahap ini bertujuan untuk mengurangi penggunaan atribut
yang tidak memberikan informasi penting, menghindari
penggunaan identitas sebagai fitur, serta mengurangi
potensi redundansi antarvariabel.

Pemilihan fitur mempertimbangkan beberapa aspek:

1. Relevansi atribut terhadap karakteristik transaksi
   dan finansial.
2. Ketersediaan serta kualitas data.
3. Hasil analisis distribusi dan korelasi.
4. Kemungkinan hubungan yang terlalu kuat antarfitur.
5. Kesesuaian fitur dengan tujuan segmentasi.

Atribut identitas seperti TransactionID, AccountID,
DeviceID, IP Address, dan MerchantID tidak digunakan
sebagai fitur clustering.

Fitur akhir ditentukan berdasarkan hasil analisis dan
unit analisis yang telah dipilih.

### 4.3.1 Implementasi Feature Selection

In [ ]:
# Membuat fitur proporsi transaksi
df_agregasi['Proporsi_Debit'] = (
    df_agregasi['Frekuensi_Debit'] /
    df_agregasi['Frekuensi_Transaksi']
)

df_agregasi['Proporsi_ATM'] = (
    df_agregasi['Kanal_ATM'] /
    df_agregasi['Frekuensi_Transaksi']
)

df_agregasi['Proporsi_Online'] = (
    df_agregasi['Kanal_Online'] /
    df_agregasi['Frekuensi_Transaksi']
)

# Menentukan fitur yang digunakan dalam clustering
selected_features = [
    'Usia_Nasabah',
    'Rata_Rata_Nominal',
    'Frekuensi_Transaksi',
    'Rata_Rata_Durasi',
    'Rata_Rata_Login',
    'Rata_Rata_Saldo',
    'Proporsi_Debit',
    'Proporsi_ATM',
    'Proporsi_Online'
]

# Membuat dataset khusus fitur clustering
X = df_agregasi[selected_features].copy()

print("Fitur yang digunakan:")
print(X.columns.tolist())

print("\nJumlah fitur:", X.shape[1])
print("Ukuran dataset:", X.shape)

display(X.head())
print("Fitur Total_Nominal_Transaksi tidak dipilih karena memiliki potensi redundansi dengan Rata_Rata_Nominal dan Frekuensi_Transaksi.")

### 4.3.2 Feature Scaling

Feature Scaling merupakan proses menyamakan rentang
skala antarfitur agar atribut dengan nilai numerik besar
tidak mendominasi perhitungan jarak.

Tahap ini penting karena K-Means dan Hierarchical
Clustering menggunakan perhitungan jarak untuk
menentukan kemiripan antarobservasi.

Metode yang digunakan dalam penelitian ini adalah
StandardScaler.

StandardScaler mengubah nilai setiap fitur berdasarkan
rata-rata dan standar deviasi sehingga data memiliki
rata-rata mendekati 0 dan standar deviasi mendekati 1.

In [ ]:
# Inisialisasi StandardScaler
scaler = StandardScaler()

# Melakukan feature scaling
X_scaled = scaler.fit_transform(X)

# Mengubah hasil scaling menjadi DataFrame
df_scaled = pd.DataFrame(
    X_scaled,
    columns=selected_features,
    index=X.index
)

print("Hasil Feature Scaling:")
display(df_scaled.head())

In [ ]:
# Statistik sebelum scaling
print("Statistik Sebelum Scaling:")
display(X.describe().loc[['mean', 'std', 'min', 'max']])

# Statistik sesudah scaling
print("Statistik Sesudah Scaling:")
display(df_scaled.describe().loc[['mean', 'std', 'min', 'max']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sebelum scaling
sns.boxplot(data=X, ax=axes[0])
axes[0].set_title("Data Sebelum StandardScaler")
axes[0].tick_params(axis='x', rotation=45)

# Sesudah scaling
sns.boxplot(data=df_scaled, ax=axes[1])
axes[1].set_title("Data Sesudah StandardScaler")
axes[1].tick_params(axis='x', rotation=45)

print("Visualisasi Hasil Scaling")
plt.tight_layout()
plt.show()

## 4.4 Final Dataset

Tahap ini merupakan tahap akhir Data Preparation.

Dataset yang telah melalui proses cleaning, preprocessing,
dan feature selection diperiksa kembali untuk memastikan
bahwa data telah siap digunakan dalam proses clustering.

Pemeriksaan meliputi:
- Jumlah baris dan kolom.
- Atribut yang digunakan.
- Missing values.
- Tipe data.
- Hasil feature scaling.

Dataset akhir kemudian disimpan dan digunakan sebagai
input untuk pemodelan menggunakan K-Means dan
Hierarchical Clustering.

In [ ]:
df_final = df_scaled.copy()
X_final = df_final

print("Ukuran Final Dataset:", df_final.shape)
print("Fitur yang digunakan:", df_final.columns.tolist())
print("Jumlah missing value:", df_final.isnull().sum().sum())
print("\nContoh Final Dataset:")
display(df_final.head())

---
# 5. Modeling

## 5.1 Pemodelan K-Means Clustering
K-Means merupakan algoritma clustering yang membagi data ke dalam sejumlah cluster berdasarkan kedekatan jarak setiap observasi terhadap pusat cluster (centroid).

Algoritma ini bekerja secara iteratif dengan menentukan centroid awal, mengelompokkan setiap observasi ke centroid terdekat, kemudian memperbarui posisi centroid hingga proses mencapai kondisi konvergen.

Pada penelitian ini, K-Means digunakan untuk membentuk kelompok berdasarkan kemiripan pola transaksi dan karakteristik finansial.

In [ ]:
X_kmeans = X_final

print("Ukuran dataset:", X_kmeans.shape)
print("Fitur yang digunakan:")
print(X_kmeans.columns.tolist())
X_kmeans.head()

### 5.1.1 Menentukan Jumlah Cluster Menggunakan Elbow Method

Pada penelitian ini, Elbow Method digunakan untuk menguji beberapa kemungkinan jumlah cluster dan memperoleh rentang jumlah cluster yang dapat dipertimbangkan pada proses pemodelan K-Means.

Hasil yang akan dianalisis:
- Grafik Elbow Method.
- Perubahan nilai inertia pada setiap jumlah cluster.
- Perkiraan jumlah cluster optimal berdasarkan titik siku.

In [ ]:
inertia = []
K_range = range(1, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_kmeans)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertia, marker='o')
plt.xlabel('Jumlah Cluster (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method K-Means')
plt.show()

### 5.1.2 Implementasi Algoritma K-Means
Setelah memperoleh perkiraan jumlah cluster melalui Elbow Method, algoritma K-Means diterapkan menggunakan jumlah cluster yang telah dipilih.

Pada tahap ini, hasil clustering disimpan untuk digunakan dalam proses evaluasi dan analisis karakteristik masing-masing cluster.

Hasil yang akan dianalisis:
- Jumlah cluster yang digunakan.
- Jumlah observasi pada setiap cluster.
- Distribusi anggota pada masing-masing cluster.

In [ ]:
# Menghitung Silhouette Score untuk K=3, 4, dan 5

hasil_silhouette = []

for k in range(3, 6):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_kmeans)
    score = silhouette_score(X_kmeans, labels)
    hasil_silhouette.append([k, score])

hasil_kmeans_df = pd.DataFrame(
    hasil_silhouette,
    columns=['Jumlah Cluster (K)', 'Silhouette Score']
)

print("Perbandingan Silhouette Score:")
display(hasil_kmeans_df)

# Memilih K dengan Silhouette Score tertinggi sebagai kandidat.
best_k = int(hasil_kmeans_df.loc[hasil_kmeans_df['Silhouette Score'].idxmax(), 'Jumlah Cluster (K)'])
print("K terbaik sementara:", best_k)

Pemilihan akhir tetap memperhatikan grafik Elbow dan profil cluster.

### 5.1.3 Evaluasi K-Means Menggunakan Silhouette Score
Silhouette Score digunakan untuk mengevaluasi kualitas hasil clustering berdasarkan tingkat kedekatan suatu observasi dengan cluster tempatnya berada dibandingkan dengan cluster lainnya.

Semakin tinggi nilai Silhouette Score, semakin baik pemisahan dan kekompakan cluster yang terbentuk.

Evaluasi dilakukan untuk mengetahui kualitas hasil clustering K-Means berdasarkan jumlah cluster yang telah dipilih.

Hasil yang akan dianalisis:
- Nilai Silhouette Score keseluruhan.
- Interpretasi kualitas pemisahan cluster.
- Apakah hasil clustering menunjukkan pemisahan yang cukup baik.

In [ ]:
# Membuat model K-Means dengan jumlah cluster terpilih

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
label_kmeans = kmeans.fit_predict(X_kmeans)

# Menambahkan hasil cluster ke data akun
df_hasil_kmeans = df_agregasi.copy()
df_hasil_kmeans['Cluster_KMeans'] = label_kmeans

# Menghitung Silhouette Score model akhir
score_kmeans = silhouette_score(X_kmeans, label_kmeans)

print("Jumlah cluster:", best_k)
print("Silhouette Score K-Means:", round(score_kmeans, 4))
print("\nJumlah anggota setiap cluster:")
print(df_hasil_kmeans['Cluster_KMeans'].value_counts().sort_index())

### 5.1.4 Visualisasi Hasil Clustering K-Means
Visualisasi dilakukan untuk memberikan gambaran mengenai persebaran observasi berdasarkan cluster yang telah terbentuk.

Hasil reduksi dimensi digunakan untuk memvisualisasikan persebaran cluster dalam bentuk scatter plot.

Hasil yang akan dianalisis:
- Persebaran observasi pada setiap cluster.
- Tingkat pemisahan antarcluster secara visual.
- Kemungkinan adanya tumpang tindih antarcluster.

In [ ]:
# Visualisasi hasil clustering K-Means
# Dua fitur dipilih agar hasil cluster mudah dilihat.

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_hasil_kmeans,
    x='Rata_Rata_Nominal',
    y='Rata_Rata_Saldo',
    hue='Cluster_KMeans',
    palette='viridis',
    s=60
)
plt.title('Hasil Clustering K-Means')
plt.xlabel('Rata-rata Nominal Transaksi')
plt.ylabel('Rata-rata Saldo')
plt.show()

### 5.1.5 Profil dan Interpretasi Cluster K-Means
Profiling cluster dilakukan untuk memahami karakteristik setiap kelompok yang terbentuk berdasarkan algoritma K-Means.

Analisis dilakukan dengan menghitung nilai rata-rata atau median setiap fitur pada masing-masing cluster. Hasil tersebut kemudian dibandingkan untuk mengidentifikasi perbedaan karakteristik antarcluster.

Atribut yang dianalisis mencakup pola transaksi dan karakteristik finansial, seperti nominal transaksi, frekuensi transaksi, durasi transaksi, jumlah percobaan login, saldo rekening, dan usia nasabah sesuai fitur yang digunakan.

Hasil yang akan dianalisis:
- Jumlah anggota setiap cluster.
- Rata-rata atau median setiap fitur pada masing-masing cluster.
- Perbedaan karakteristik antarcluster.
- Interpretasi dan pemberian nama pada setiap cluster.

In [ ]:
# Profil cluster K-Means
# Menampilkan rata-rata fitur asli agar karakteristik tiap cluster mudah dibaca.

fitur_profil = [
    'Usia_Nasabah',
    'Rata_Rata_Nominal',
    'Frekuensi_Transaksi',
    'Rata_Rata_Durasi',
    'Rata_Rata_Login',
    'Rata_Rata_Saldo',
    'Proporsi_Debit',
    'Proporsi_ATM',
    'Proporsi_Online'
]

print("Jumlah anggota setiap cluster:")
print(df_hasil_kmeans['Cluster_KMeans'].value_counts().sort_index())

print("\nRata-rata fitur setiap cluster:")
profil_cluster = df_hasil_kmeans.groupby('Cluster_KMeans')[fitur_profil].mean().round(2)
display(profil_cluster)

print("\nDistribusi pekerjaan pada setiap cluster (untuk membantu interpretasi):")
display(pd.crosstab(
    df_hasil_kmeans['Cluster_KMeans'],
    df_hasil_kmeans['Pekerjaan_Nasabah']
))

---
# 6. Hierarchical Clustering
## 6.1 Pemodelan Agglomerative Hierarchical Clustering
Pada penelitian ini digunakan pendekatan Agglomerative Hierarchical Clustering, yaitu pendekatan bottom-up yang dimulai dengan menganggap setiap observasi sebagai satu cluster tersendiri. Selanjutnya, dua cluster yang paling dekat akan digabungkan secara bertahap hingga terbentuk struktur hierarki.

Salah satu karakteristik Hierarchical Clustering adalah penggunaan metode linkage untuk menentukan cara menghitung jarak antarcluster.

Beberapa metode linkage yang digunakan dalam penelitian ini adalah Single, Complete, Average, dan Ward.

## 6.2 Perbandingan Metode Linkage
### 6.2.1 Implementasi Agglomerative Clustering dengan Beberapa Linkage
Pada tahap ini, Agglomerative Clustering diterapkan menggunakan beberapa metode linkage, yaitu Single, Complete, Average, dan Ward.

Setiap metode diuji menggunakan jumlah cluster yang sama agar perbandingan kualitas hasil clustering dapat dilakukan secara lebih adil.

Hasil dari setiap pemodelan berupa label cluster yang digunakan untuk menghitung nilai evaluasi dan menganalisis karakteristik pengelompokan.

Hasil yang akan dianalisis:
- Label cluster dari setiap metode linkage.
- Jumlah anggota setiap cluster.
- Perbedaan hasil pengelompokan antarmetode linkage.

In [ ]:
# Agglomerative Clustering untuk beberapa linkage

# 1. Ward Linkage
agg_ward = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward",
    metric="euclidean"
)
labels_ward = agg_ward.fit_predict(X_scaled)


# 2. Average Linkage
agg_avg = AgglomerativeClustering(
    n_clusters=3,
    linkage="average",
    metric="euclidean"
)
labels_avg = agg_avg.fit_predict(X_scaled)


# 3. Complete Linkage
agg_comp = AgglomerativeClustering(
    n_clusters=3,
    linkage="complete",
    metric="euclidean"
)
labels_comp = agg_comp.fit_predict(X_scaled)


# 4. Single Linkage
agg_single = AgglomerativeClustering(
    n_clusters=3,
    linkage="single",
    metric="euclidean"
)
labels_single = agg_single.fit_predict(X_scaled)


# Menampilkan 10 label pertama dari setiap metode
print("Label 10 sampel pertama:")

print("Ward     :", labels_ward[:10])
print("Average  :", labels_avg[:10])
print("Complete :", labels_comp[:10])
print("Single   :", labels_single[:10])

### 6.2.2 Perbandingan Silhouette Score Antar-Linkage
Silhouette Score digunakan untuk membandingkan kualitas hasil clustering dari setiap metode linkage.

Perhitungan dilakukan menggunakan jumlah cluster yang sama pada setiap metode sehingga perbandingan dapat berfokus pada pengaruh perbedaan metode linkage.

Hasil evaluasi disajikan dalam bentuk tabel atau grafik agar metode dengan kualitas clustering terbaik dapat diidentifikasi.

Hasil yang akan dianalisis:
- Nilai Silhouette Score Single Linkage.
- Nilai Silhouette Score Complete Linkage.
- Nilai Silhouette Score Average Linkage.
- Nilai Silhouette Score Ward Linkage.
- Metode linkage dengan nilai Silhouette Score tertinggi.

In [ ]:
# Menghitung dan membandingkan Silhouette Score

linkage_results = {
    "Ward": labels_ward,
    "Average": labels_avg,
    "Complete": labels_comp,
    "Single": labels_single
}

silhouette_results = {}

for name, labels in linkage_results.items():
    score = silhouette_score(X_scaled, labels)
    silhouette_results[name] = score


# Membuat tabel perbandingan
df_silhouette = pd.DataFrame(
    list(silhouette_results.items()),
    columns=["Metode Linkage", "Silhouette Score"]
)

df_silhouette = df_silhouette.sort_values(
    by="Silhouette Score",
    ascending=False
).reset_index(drop=True)

print("\nPerbandingan Silhouette Score:")
display(df_silhouette)

### 6.2.3 Visualisasi Dendrogram Menggunakan Ward Linkage
Dendrogram merupakan visualisasi berbentuk pohon yang menggambarkan proses penggabungan cluster pada Hierarchical Clustering.

Pada penelitian ini, dendrogram dibuat menggunakan metode Ward Linkage untuk mengamati struktur hierarki dan jarak penggabungan antarcluster.

Sumbu horizontal menunjukkan observasi atau kelompok observasi, sedangkan sumbu vertikal menunjukkan jarak atau tingkat ketidakmiripan ketika dua cluster digabungkan.

Perubahan jarak penggabungan yang cukup besar dapat menjadi indikasi adanya kelompok yang relatif berbeda sehingga dapat dipertimbangkan dalam menentukan jumlah cluster.

Hasil yang akan dianalisis:
- Struktur hierarki penggabungan cluster.
- Perubahan jarak penggabungan.
- Tinggi penggabungan yang menunjukkan perbedaan antarkelompok.

In [ ]:
# Dendrogram Ward Linkage

Z = linkage(
    X_scaled,
    method="ward"
)

plt.figure(figsize=(15, 7))

dendrogram(
    Z,
    truncate_mode="lastp",
    p=15,
    leaf_rotation=0,
    leaf_font_size=9
)

plt.title("Dendrogram - Ward Linkage")
plt.xlabel("Cluster / Data")
plt.ylabel("Jarak Penggabungan")
plt.tight_layout()
plt.show()

### 6.2.4 Penentuan Jumlah Cluster melalui Pemotongan Dendrogram

Setelah struktur hierarki terbentuk, langkah berikutnya adalah menentukan jumlah cluster melalui proses pemotongan dendrogram (*dendrogram cutting*).

Pemotongan dilakukan dengan menentukan batas jarak (*cut-off distance*) pada sumbu vertikal dendrogram. Batas tersebut digunakan untuk memisahkan kelompok-kelompok yang terbentuk berdasarkan struktur hierarki.

Pada penelitian ini, pemotongan dendrogram diarahkan untuk memperoleh tiga cluster agar konsisten dengan jumlah cluster yang digunakan pada pemodelan K-Means dan Agglomerative Clustering sebelumnya.

Batas pemotongan ditentukan berdasarkan ketinggian dua penggabungan yang mengapit jumlah cluster target. Selanjutnya, hasil pemotongan diperiksa untuk memastikan jumlah cluster yang terbentuk sesuai dengan target.

Hasil yang akan dianalisis:
- Ketinggian penggabungan yang membentuk tiga cluster.
- Batas jarak pemotongan dendrogram.
- Jumlah cluster yang dihasilkan setelah pemotongan.
- Kesesuaian hasil pemotongan dengan jumlah cluster yang ditargetkan.

In [ ]:
# Menentukan pemotongan dendrogram Ward

from scipy.cluster.hierarchy import fcluster

# Menentukan jumlah cluster yang diinginkan
k_target = 3

# Jumlah seluruh observasi
n = X_scaled.shape[0]

# Menentukan indeks penggabungan yang mengapit 3 cluster
idx_masuk_k = n - k_target - 1
idx_keluar_k = idx_masuk_k + 1

# Menghitung batas pemotongan dendrogram
max_d = (
    Z[idx_masuk_k, 2] +
    Z[idx_keluar_k, 2]
) / 2

print(f"Ketinggian merge pembentuk {k_target} cluster:",
      round(Z[idx_masuk_k, 2], 4))

print(f"Ketinggian merge berikutnya:",
      round(Z[idx_keluar_k, 2], 4))

print(f"Batas pemotongan dendrogram:",
      round(max_d, 4))


# Memotong dendrogram berdasarkan jarak
labels_dendrogram = fcluster(
    Z,
    t=max_d,
    criterion="distance"
)

# Memeriksa jumlah cluster yang terbentuk
jumlah_cluster = len(np.unique(labels_dendrogram))

print("\nJumlah cluster hasil pemotongan dendrogram:",
      jumlah_cluster)


# Visualisasi dendrogram dengan garis pemotongan
plt.figure(figsize=(15, 7))

dendrogram(
    Z,
    color_threshold=max_d,
    truncate_mode="lastp",
    p=15
)

plt.axhline(
    y=max_d,
    color="black",
    linestyle="--",
    label=f"Cut-off = {max_d:.4f}"
)

plt.title("Dendrogram Ward dengan Garis Pemotongan")
plt.xlabel("Cluster / Data")
plt.ylabel("Jarak Penggabungan")
plt.legend()
plt.tight_layout()
plt.show()

### 6.2.5 Visualisasi Cluster Menggunakan Heatmap
Heatmap digunakan untuk menampilkan perbedaan karakteristik fitur pada setiap cluster dalam bentuk representasi warna.

Visualisasi dilakukan dengan menghitung nilai rata-rata setiap fitur pada masing-masing cluster. Agar perbedaan antarfitur lebih mudah diamati, nilai rata-rata dapat distandardisasi terlebih dahulu sebelum ditampilkan.

Heatmap membantu mengidentifikasi fitur yang memiliki nilai relatif tinggi atau rendah pada setiap cluster.

Hasil yang akan dianalisis:
- Pola karakteristik fitur pada setiap cluster.
- Fitur yang paling membedakan antarcluster.
- Kesamaan dan perbedaan karakteristik kelompok.

### 6.2.6 Profil dan Interpretasi Cluster Hierarchical
Profiling dilakukan untuk memahami karakteristik kelompok yang terbentuk berdasarkan Agglomerative Hierarchical Clustering.

Analisis dilakukan dengan menghitung nilai rata-rata atau median setiap fitur pada masing-masing cluster. Hasil tersebut digunakan untuk mengidentifikasi pola transaksi dan karakteristik finansial yang dominan.

Setiap cluster kemudian diberikan interpretasi berdasarkan karakteristik yang ditemukan.

Hasil profiling digunakan untuk menjelaskan makna kelompok yang terbentuk serta potensi pemanfaatannya dalam strategi layanan perbankan.

Hasil yang akan dianalisis:
- Jumlah anggota setiap cluster.
- Rata-rata atau median setiap fitur.
- Perbedaan karakteristik antarcluster.
- Interpretasi dan pemberian nama cluster.

In [ ]:
# Profil Cluster Hierarchical - Ward

# Menggunakan data hasil agregasi rekening
df_hierarchical = df_agregasi.copy()

# Menambahkan label cluster Ward
df_hierarchical["Cluster_Hierarchical"] = labels_ward


# Menghitung rata-rata setiap fitur berdasarkan cluster
profil_hierarchical = df_hierarchical.groupby(
    "Cluster_Hierarchical"
)[selected_features].mean()


# Menghitung jumlah anggota setiap cluster
profil_hierarchical["Jumlah_Anggota"] = (
    df_hierarchical
    .groupby("Cluster_Hierarchical")
    .size()
)


# Menghitung persentase anggota setiap cluster
profil_hierarchical["Persentase_Anggota"] = (
    profil_hierarchical["Jumlah_Anggota"] /
    len(df_hierarchical) * 100
)


# Menampilkan hasil profiling
print("Profil Cluster Hierarchical - Ward:")
display(profil_hierarchical.round(3))

---
# 7. Evaluation
## 7.1 Perbandingan Hasil K-Means dan Hierarchical Clustering
Tahap evaluasi dilakukan untuk membandingkan hasil clustering yang diperoleh dari algoritma K-Means dan Hierarchical Clustering.

Perbandingan dilakukan menggunakan dataset, fitur, dan skala data yang sama agar hasil evaluasi dapat dibandingkan secara objektif.

Evaluasi tidak hanya berfokus pada nilai Silhouette Score, tetapi juga mempertimbangkan distribusi anggota cluster, karakteristik setiap kelompok, dan kemudahan interpretasi hasil.

## 7.2 Evaluasi Kualitas dan Interpretabilitas Cluster
Selain membandingkan nilai Silhouette Score, evaluasi dilakukan dengan memperhatikan apakah cluster yang terbentuk memiliki karakteristik yang cukup berbeda dan dapat dijelaskan secara logis.

Cluster yang memiliki nilai evaluasi baik belum tentu secara langsung menghasilkan segmentasi yang bermanfaat apabila karakteristik setiap kelompok sulit dibedakan.

Oleh karena itu, interpretasi hasil dilakukan dengan menghubungkan karakteristik cluster terhadap tujuan penelitian, yaitu memahami pola transaksi dan karakteristik finansial.

Evaluasi ini juga mempertimbangkan potensi bias akibat fitur tertentu yang terlalu dominan dalam membentuk cluster.

## 7.3 Penentuan Model Terbaik
Penentuan model terbaik dilakukan berdasarkan hasil perbandingan kedua algoritma dengan mempertimbangkan kualitas clustering dan relevansi hasil terhadap tujuan penelitian.

Model terbaik tidak hanya ditentukan berdasarkan nilai Silhouette Score tertinggi, tetapi juga berdasarkan:
- Kualitas pemisahan dan kekompakan cluster.
- Distribusi anggota cluster.
- Perbedaan karakteristik antarcluster.
- Kemudahan interpretasi hasil.
- Kesesuaian hasil segmentasi dengan tujuan penelitian.
Berdasarkan hasil evaluasi tersebut, ditentukan algoritma yang paling sesuai untuk membentuk segmentasi berdasarkan pola transaksi dan karakteristik finansial.

## 7.4 Kesimpulan Hasil Modeling dan Evaluation
Bagian ini berisi rangkuman hasil pemodelan dan evaluasi yang telah dilakukan.

Kesimpulan mencakup jumlah cluster terbaik, karakteristik utama setiap cluster, perbandingan performa K-Means dan Hierarchical Clustering, serta algoritma yang dipilih sebagai model terbaik.

Hasil akhir digunakan untuk menjawab rumusan masalah penelitian mengenai pembentukan segmentasi, interpretasi karakteristik kelompok, dan perbandingan kualitas kedua algoritma.